# Montandon: Floods

This notebook fetches recent flood data from Montandon, the global crisis data bank, filtering for severity, and visualizes them on a map.

In [45]:
import os
import pandas as pd
from pystac_client import Client
import geopandas as gpd
from shapely.geometry import Point, Polygon, shape
from lonboard import viz

from datetime import datetime, timedelta, timezone

## Connect to Montandon STAC

Montandon is exposed as a STAC collection, but requires authentication.

In [46]:
STAC_API_URL = "https://montandon-eoapi-stage.ifrc.org/stac"
API_TOKEN = os.getenv('MONTANDON_API_TOKEN')

In [47]:
auth_headers = {"Authorization": f"Bearer {API_TOKEN}"}

### Check Auth

In [48]:
# Connect to STAC API with authentication
try:
    client = Client.open(STAC_API_URL, headers=auth_headers)
    print(f"\n[OK] Connected to: {STAC_API_URL}")
    print(f"[OK] API Title: {client.title}")
    print(f"[OK] Authentication: Bearer Token (OpenID Connect)")
except Exception as e:
    print(f"\n[ERROR] Authentication failed: {e}")


[OK] Connected to: https://montandon-eoapi-stage.ifrc.org/stac
[OK] API Title: Montandon STAC API
[OK] Authentication: Bearer Token (OpenID Connect)


In [49]:
client

<Client id=montandon-eoapi>

## Fetch Recent Flood Impacts

We search for flood impact records directly, and then fetch hazard data based on the monty correlation ID.

Floods are searched for by the UNDRR hazard type `MH0600`, [which are mapped](https://github.com/IFRCGo/monty-stac-extension/tree/main/docs/model/sources/GDACS#mapping-from-gdacs-event-type-to-hazard-profile) to GDACS flood event `FL`. 

In [50]:
now = datetime.now(timezone.utc)
start = now - timedelta(days=90)  # adjust to change the search window
items_max = 5000 # Keeps max items consistent across searches

_Note that this search takes several minutes to run with `items_max = 5000`._

In [51]:
search = client.search(
    collections=["gdacs-impacts"],
    datetime=f"{start.isoformat()}/{now.isoformat()}",
    max_items=items_max,
)

# MH0600 is the standard UNDRR flood hazard code
flood_impacts = [
    item for item in search.items()
    if 'MH0600' in item.properties.get('monty:hazard_codes', [])
]
print(f"Found {len(flood_impacts)} flood impact records")

Found 593 flood impact records


Each impact below returns a row for impact type and its value.

In [52]:
def parse_impact(item):
    detail = item.properties.get('monty:impact_detail', {})
    return {
        'monty_corr_id': item.properties.get('monty:corr_id'),
        'title': item.properties.get('title'),
        'country_codes': ', '.join(item.properties.get('monty:country_codes', [])),
        'impact_type': detail.get('type'),
        'impact_value': detail.get('value'),
    }

impacts_df = pd.DataFrame([parse_impact(item) for item in flood_impacts])
impacts_df

,monty_corr_id,title,country_codes,impact_type,impact_value
0,20260503-UGA-816471-MH0600-1-GCDB,Flood in Uganda,UGA,damaged,50
1,20260503-UGA-816471-MH0600-1-GCDB,Flood in Uganda,UGA,relocated,50
2,20260501-UZB-1173054-MH0600-1-GCDB,Flood in Uzbekistan,UZB,affected_total,1
3,20260428-PER-720521-MH0600-4-GCDB,Flood in Peru,PER,damaged,4
4,20260428-PER-720521-MH0600-4-GCDB,Flood in Peru,PER,affected_total,30
...,...,...,...,...,...
588,20260203-BRA-733313-MH0600-22-GCDB,Flood in Brazil,BRA,damaged,20
589,20260203-BRA-733313-MH0600-21-GCDB,Flood in Brazil,BRA,damaged,20
590,20260203-BRA-733313-MH0600-20-GCDB,Flood in Brazil,BRA,damaged,20
591,20260203-BRA-733313-MH0600-19-GCDB,Flood in Brazil,BRA,damaged,20


We pivot the rows, so that each event, matched with the Monty correlation id `monty_corr_id`, is a single row, with columns being the values of potential impacts.

In [53]:
# Pivot: one row per event, impact types as columns
impacts_pivot = impacts_df.pivot_table(
    index='monty_corr_id',
    columns='impact_type',
    values='impact_value',
    aggfunc='sum'
).reset_index()

# Re-attach metadata — take first value per corr_id
metadata = (
    impacts_df[['monty_corr_id', 'title', 'country_codes']]
    .drop_duplicates('monty_corr_id')
)
impacts_pivot = impacts_pivot.merge(metadata, on='monty_corr_id')

print(f"{len(impacts_pivot)} flood events with impact data")
impacts_pivot

74 flood events with impact data


,monty_corr_id,affected_total,assisted,damaged,death,injured,missing,relocated,title,country_codes
0,20260203-BRA-733313-MH0600-18-GCDB,NaN,NaN,20.0,NaN,NaN,NaN,NaN,Flood in Brazil,BRA
1,20260203-BRA-733313-MH0600-19-GCDB,NaN,NaN,28.0,NaN,NaN,NaN,10.0,Flood in Brazil,BRA
2,20260203-BRA-733313-MH0600-20-GCDB,NaN,NaN,48.0,NaN,NaN,NaN,10.0,Flood in Brazil,BRA
3,20260203-BRA-733313-MH0600-21-GCDB,NaN,NaN,48.0,NaN,NaN,NaN,10.0,Flood in Brazil,BRA
4,20260203-BRA-733313-MH0600-22-GCDB,2000.0,NaN,48.0,1.0,NaN,NaN,40.0,Flood in Brazil,BRA
...,...,...,...,...,...,...,...,...,...,...
69,20260501-ZAF-564431-MH0600-1-GCDB,3000.0,2.0,NaN,NaN,NaN,NaN,500.0,Flood in South Africa,ZAF
70,20260502-DEU-1284349-MH0600-1-GCDB,NaN,NaN,40.0,NaN,NaN,NaN,NaN,Flood in Germany,DEU
71,20260502-DEU-1284349-MH0600-2-GCDB,NaN,NaN,40.0,NaN,NaN,NaN,NaN,Flood in Germany,DEU
72,20260503-MNG-796699-MH0600-1-GCDB,81.0,NaN,NaN,NaN,NaN,NaN,NaN,"Flood in Kenya, Mongolia",KEN


## Get Flood Footprints

Use the correlation IDs from the impact records to fetch the matching flood footprint polygons from GDACS hazards.

In [54]:
corr_id_set = set(impacts_pivot['monty_corr_id'])

search = client.search(
    collections=["gdacs-hazards"],
    max_items=items_max,
)

hazard_items = [
    item for item in search.items()
    if item.properties.get('monty:corr_id') in corr_id_set
]
print(f"Matched {len(hazard_items)} hazard footprints")

Matched 74 hazard footprints


In [55]:
hazards_gdf = gpd.GeoDataFrame(
    [{
        'monty_corr_id': item.properties['monty:corr_id'],
        'severity_label': item.properties.get('monty:hazard_detail', {}).get('severity_label'),
        'geometry': shape(item.geometry),
    } for item in hazard_items],
    geometry='geometry',
    crs='EPSG:4326',
)

# Combine footprints with impact data
floods = hazards_gdf.merge(impacts_pivot, on='monty_corr_id', how='left')
floods

,monty_corr_id,severity_label,geometry,affected_total,assisted,damaged,death,injured,missing,relocated,title,country_codes
0,20260503-UGA-816471-MH0600-1-GCDB,Green,"POLYGON ((33.9823 0.5751, 34.2766 0.6396, 34.3...",NaN,NaN,50.0,NaN,NaN,NaN,50.0,Flood in Uganda,UGA
1,20260503-MNG-796699-MH0600-1-GCDB,Green,"POLYGON ((38.9597 -1.7044, 39.1792 -2.0125, 39...",81.0,NaN,NaN,NaN,NaN,NaN,NaN,"Flood in Kenya, Mongolia",KEN
2,20260502-DEU-1284349-MH0600-2-GCDB,Green,"MULTIPOLYGON (((9.3171 52.7592, 9.6989 52.6001...",NaN,NaN,40.0,NaN,NaN,NaN,NaN,Flood in Germany,DEU
3,20260502-DEU-1284349-MH0600-1-GCDB,Green,"MULTIPOLYGON (((6.5003 51.2899, 6.674 51.3886,...",NaN,NaN,40.0,NaN,NaN,NaN,NaN,Flood in Germany,DEU
4,20260501-UZB-1173054-MH0600-1-GCDB,Green,"POLYGON ((70.5734 40.4324, 70.6308 40.429, 70....",1.0,NaN,90.0,NaN,NaN,NaN,NaN,Flood in Uzbekistan,UZB
...,...,...,...,...,...,...,...,...,...,...,...,...
69,20260203-BRA-733313-MH0600-22-GCDB,Green,"POLYGON ((-50.2076 -9.839, -50.0479 -9.3082, -...",2000.0,NaN,48.0,1.0,NaN,NaN,40.0,Flood in Brazil,BRA
70,20260203-BRA-733313-MH0600-21-GCDB,Green,"POLYGON ((-47.679 -13.467, -47.6331 -13.1018, ...",NaN,NaN,48.0,NaN,NaN,NaN,10.0,Flood in Brazil,BRA
71,20260203-BRA-733313-MH0600-20-GCDB,Green,"POLYGON ((-54.2874 -24.0686, -53.607 -22.951, ...",NaN,NaN,48.0,NaN,NaN,NaN,10.0,Flood in Brazil,BRA
72,20260203-BRA-733313-MH0600-19-GCDB,Green,"POLYGON ((-47.8561 -25.4896, -46.5804 -24.675,...",NaN,NaN,28.0,NaN,NaN,NaN,10.0,Flood in Brazil,BRA


In [56]:
viz(floods)

## Manywidgets

In [57]:
from manywidgets import Stat, Grid

In [59]:
Grid(
    Stat(label="Population affected", value=impacts_pivot.affected_total.sum()),
    Stat(label="People Killed", value=impacts_pivot.death.sum()),
    Stat(label="People Injured", value=impacts_pivot.injured.sum()),
    Stat(label="People Displaced", value=impacts_pivot.relocated.sum()),
    Stat(label="Buildings Damaged", value=impacts_pivot.damaged.sum()),
    # Stat(label="Buildings Destroyed", value=impacts_pivot.destroyed.sum()),
    Stat(label="Countries Affected", value=impacts_pivot.country_codes.count()),
    columns=3, gap="6px",
)